# Heavy-Tailed ABMs: Cont-Bouchaud & OFC

This notebook reproduces power-law experiments from two agent-based models:

1. **Cont-Bouchaud (CB)**: herd behaviour and fat tails in financial markets.
2. **Olami-Feder-Christensen (OFC)**: self-organised criticality in seismic faults.

Sections:
1. Setup & imports
2. CB simulation demo
3. CB phase diagram
4. CB calibration (MLE + ABC)
5. OFC simulation demo
6. OFC phase diagram
7. OFC calibration (MLE + ABC)

## 1. Setup & imports

In [1]:
import os
import sys
import warnings

warnings.filterwarnings('ignore')

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.dirname('.'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless backend; switch to 'inline' in Jupyter
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

from models.cont_bouchaud import simulate_cb
from models.ofc import simulate_ofc
from utils.powerlaw_fit import fit_powerlaw, gutenberg_richter_b
from utils.plotting import (
    plot_ccdf, plot_return_series, 
    FIGURES_DIR, _apply_style, _ensure_figures_dir)

_ensure_figures_dir()

## 2. Cont-Bouchaud simulation demo

Single run at `p=0.59` (near the percolation threshold), `L=64`, 10 000 steps.

In [ ]:
# CB single run
CB_L, CB_P, CB_A, CB_STEPS = 64, 0.59, 0.01, 10_000
returns_cb = simulate_cb(CB_L, CB_P, CB_A, CB_STEPS, seed=42)

print(f'CB returns: mean={returns_cb.mean():.4f}, std={returns_cb.std():.4f}')
print(f'Kurtosis: {float(np.mean((returns_cb - returns_cb.mean())**4) / returns_cb.std()**4):.2f}')

In [ ]:
# Return time series
save_ts = os.path.join(FIGURES_DIR, 'cb_demo_timeseries.pdf')
plot_return_series(returns_cb, title='CB returns (p=0.59, L=64)', save_path=save_ts)
print('Saved:', save_ts)

In [ ]:
# CCDF of absolute returns
_apply_style()
fig, ax = plt.subplots(figsize=(3.5, 2.8))

abs_ret = np.abs(returns_cb[returns_cb != 0])
fit_res = fit_powerlaw(abs_ret)
print(f"Power-law alpha = {fit_res['alpha']:.3f} (xmin={fit_res['xmin']:.4f})")
print(f"vs lognormal: R={fit_res['R_lognormal']:.3f}, p={fit_res['p_lognormal']:.3f}")

plot_ccdf(abs_ret, label='CB |returns|', ax=ax, color='steelblue', fit_result=fit_res)
ax.set_title(f'CB CCDF (p={CB_P}, L={CB_L})')

save_ccdf = os.path.join(FIGURES_DIR, 'cb_demo_ccdf.pdf')
fig.tight_layout()
fig.savefig(save_ccdf)
plt.close(fig)
print('Saved:', save_ccdf)

## 3. CB Phase Diagram

Grid search over `p ∈ [0.40, 0.70]` and `L ∈ {32, 64, 128}`.

In [ ]:
from experiments.cb_phase_diagram import (
    run_cb_phase_diagram, plot_cb_phase_diagram,
    P_GRID, L_VALUES as CB_L_VALUES)

cb_exponents = run_cb_phase_diagram()
plot_cb_phase_diagram(cb_exponents)

## 4. CB Calibration on CAC 40 and S&P 500

### 4a. MLE calibration

In [ ]:
from data.download_data import download_stock_returns
from experiments.cb_calibration import run_mle_calibration as cb_mle

print('Downloading stock returns ...')
returns_dict = download_stock_returns(
    tickers=['^FCHI', '^GSPC'],
    start='2000-01-01',
    end='2024-01-01')

for ticker, rets in returns_dict.items():
    abs_ret = np.abs(rets.dropna().values)
    abs_ret = abs_ret[abs_ret > 0]
    res = fit_powerlaw(abs_ret)
    alpha_emp = res['alpha']
    print(f'\n{ticker}: empirical alpha = {alpha_emp:.4f}')
    cb_mle(alpha_emp, ticker)

### 4b. ABC calibration

In [ ]:
from experiments.cb_calibration import run_abc_calibration as cb_abc

for ticker, rets in returns_dict.items():
    abs_ret = np.abs(rets.dropna().values)
    abs_ret = abs_ret[abs_ret > 0]
    res = fit_powerlaw(abs_ret)
    alpha_emp = res['alpha']
    print(f'\nABC calibration for {ticker} (alpha_emp={alpha_emp:.4f}) ...')
    try:
        cb_abc(alpha_emp, ticker)
    except Exception as e:
        print(f'  Skipped: {e}')

## 5. OFC Simulation Demo

Single run with `alpha_ofc=0.20`, `L=64`, `n_events=20_000`, `seed=42`.

In [2]:
# OFC demo parameters (fast: <10 s)
OFC_L = 64
OFC_ALPHA = 0.20
OFC_EVENTS = 20_000

sizes_ofc = simulate_ofc(OFC_L, OFC_ALPHA, OFC_EVENTS, seed=42)

print(f"Avalanche sizes: min={sizes_ofc.min()}, max={sizes_ofc.max()}")
print(f"Mean size: {sizes_ofc.mean():.2f}")
print(f"Fraction size-1: {(sizes_ofc == 1).mean():.3f}")

# Plot 1: time series (first 2 000 events, semilogy)
_apply_style()
fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.semilogy(sizes_ofc[:2000], linewidth=0.4, color="darkorange")
ax.set_xlabel("Event index")
ax.set_ylabel("Avalanche size")
ax.set_title(rf"OFC avalanche time series ($L={OFC_L}$, $\alpha_{{\mathrm{{OFC}}}}={OFC_ALPHA}$)")
fig.tight_layout()
save_ts_ofc = os.path.join(FIGURES_DIR, "ofc_demo_timeseries.pdf")
fig.savefig(save_ts_ofc)
plt.close(fig)
print("Saved:", save_ts_ofc)

Avalanche sizes: min=0, max=65
Mean size: 0.45
Fraction size-1: 0.080
Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_demo_timeseries.pdf


In [ ]:
# Plot 2: CCDF with power-law fit overlay
fit_res_ofc = fit_powerlaw(sizes_ofc.astype(float))
print(f"Power-law exponent alpha = {fit_res_ofc['alpha']:.3f}  (xmin={fit_res_ofc['xmin']:.1f})")

mags_ofc = np.log10(sizes_ofc[sizes_ofc > 0].astype(float))
b_ofc = gutenberg_richter_b(mags_ofc)
print(f"Gutenberg–Richter b-value = {b_ofc:.4f}")

_apply_style()
fig, ax = plt.subplots(figsize=(3.5, 2.8))
plot_ccdf(sizes_ofc.astype(float), label="Simulated avalanches", ax=ax,
          color="darkorange", fit_result=fit_res_ofc)
ax.set_title(rf"OFC model CCDF ($L={OFC_L}$, $\alpha_{{\mathrm{{OFC}}}}={OFC_ALPHA}$)")
fig.tight_layout()
save_ccdf_ofc = os.path.join(FIGURES_DIR, "ofc_demo_ccdf.pdf")
fig.savefig(save_ccdf_ofc)
plt.close(fig)
print("Saved:", save_ccdf_ofc)

Power-law exponent α = 2.591  (xmin=4.0)
Gutenberg–Richter b-value = 1.5635
Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_demo_ccdf.pdf


## 6. OFC Phase Diagram

Grid search over `alpha_ofc ∈ [0.10, 0.24]` and `L ∈ {32, 64, 128}`.  
Parameters: `N_EVENTS=50_000`, `N_SEEDS=10`.

In [2]:
from experiments.ofc_phase_diagram import run_ofc_phase_diagram, plot_ofc_phase_diagram

b_means, b_stds = run_ofc_phase_diagram()
plot_ofc_phase_diagram(b_means, b_stds)

OFC phase diagram: 100%|██████████| 30000000/30000000 [07:14<00:00, 68984.84ev/s] 


Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_phase_diagram.pdf


'/Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_phase_diagram.pdf'

## 7. OFC Calibration on USGS Catalog

### 7a. MLE calibration

Find `alpha_ofc*` that minimises `|b_sim − b_emp|` where `b_emp` is the
Gutenberg-Richter b-value estimated from the USGS global catalog (M ≥ 4.5,
2000–2024).

In [3]:
from experiments.ofc_calibration import run_mle_calibration
from data.download_data import download_usgs_catalog
from utils.powerlaw_fit import gutenberg_richter_b
from experiments.ofc_calibration import run_mle_calibration, _ensure_figures_dir

_ensure_figures_dir()
catalog = download_usgs_catalog(min_magnitude=2.0, start="2000-01-01", end="2024-01-01")
magnitudes = catalog["magnitude"].dropna().values
magnitudes = magnitudes[magnitudes >= 4.5]
b_emp = gutenberg_richter_b(magnitudes, m_min=4.5)
print(f"Empirical b = {b_emp:.4f}")  # expected: approx 1.0
run_mle_calibration(b_emp)

Empirical b = 1.3094


OFC MLE calibration: 100%|██████████| 30000000/30000000 [12:06<00:00, 41290.67ev/s] 


  alpha_ofc* = 0.2345  |  Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_calibration_mle.pdf


(np.float64(0.23448275862068965),
 array([3.23002503, 3.05016527, 2.87375129, 2.70576705, 2.56705174,
        2.46190571, 2.34912536, 2.25244671, 2.16759215, 2.08406063,
        2.01271375, 1.94509391, 1.88618898, 1.83197825, 1.78608393,
        1.73051072, 1.70244426, 1.66680839, 1.63894672, 1.6094566 ,
        1.59200248, 1.6054748 , 1.585866  , 1.60310653, 1.6160409 ,
        1.56243057, 1.31484988, 0.74007284, 0.52062773, 0.37295332]))

### 7b. ABC calibration (skipped)

ABC-SMC calibration is **not run here** due to compute cost (hours).
To run it, execute the cell below (commented out by default).
This produces `figures/ofc_calibration_abc.pdf` with the posterior
distribution over `alpha_ofc`.

In [ ]:
# from data.download_data import download_usgs_catalog
# from utils.powerlaw_fit import gutenberg_richter_b
# from experiments.ofc_calibration import run_abc_calibration, _ensure_figures_dir

# _ensure_figures_dir()
# catalog = download_usgs_catalog(min_magnitude=2.0, start="2000-01-01", end="2024-01-01")
# magnitudes = catalog["magnitude"].dropna().values
# magnitudes = magnitudes[magnitudes >= 4.5]
# b_emp = gutenberg_richter_b(magnitudes, m_min=4.5)
# print(f"Empirical b = {b_emp:.4f}")
# run_abc_calibration(b_emp)